### DermAI Diagnostics

 ### Data Preprocessing


- Loading the cleaned dataset
- Feature selection
- Identifying target and predictor variables
- Preventing data leakage
- Train-test splitting
- Building preprocessing pipelines
- Encoding categorical variables
- Scaling numerical variables
- Preparing data for model development

### STEP 1 - Import Relevant Libraries

In [1]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

### STEP 2 — Load Dataset

In [2]:

DATA_PATH = os.path.join(
    "..",
    "data",
    "raw",
    "dermai_ml_ready.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,patient_id,age,age_group,gender,smoke,drink,pesticide,skin_cancer_history,cancer_history,has_piped_water,...,lesion_size_category,itch,grew,hurt,changed,bleed,elevation,biopsed,diagnostic,cancerous_flag
0,PAT_100,14,Young Adult,MALE,f,f,f,f,t,f,...,Medium,f,f,f,f,f,f,t,NEV,0
1,PAT_1000,65,Older Adult,MALE,f,f,f,f,f,f,...,Small,f,f,f,f,f,f,f,SEK,0
2,PAT_1006,63,Older Adult,MALE,f,f,f,f,f,f,...,Small,f,f,f,f,f,f,f,ACK,0
3,PAT_1008,64,Older Adult,MALE,f,f,f,f,f,f,...,Small,f,t,f,f,f,t,f,SEK,0
4,PAT_101,59,Older Adult,MALE,t,t,f,f,f,t,...,Large,t,t,t,f,t,t,t,BCC,1


### STEP 3 — Dataset Overview

In [3]:
print(df.shape)

df.info()

(1088, 29)
<class 'pandas.DataFrame'>
RangeIndex: 1088 entries, 0 to 1087
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   patient_id            1088 non-null   str    
 1   age                   1088 non-null   int64  
 2   age_group             1088 non-null   str    
 3   gender                1088 non-null   str    
 4   smoke                 1088 non-null   str    
 5   drink                 1088 non-null   str    
 6   pesticide             1088 non-null   str    
 7   skin_cancer_history   1088 non-null   str    
 8   cancer_history        1088 non-null   str    
 9   has_piped_water       1088 non-null   str    
 10  has_sewage_system     1088 non-null   str    
 11  background_father     1088 non-null   str    
 12  background_mother     1088 non-null   str    
 13  lesion_id             1088 non-null   int64  
 14  fitspatrick           1088 non-null   int64  
 15  region               

### STEP 4 — Drop Non-Predictive Columns

In [4]:
# step 4a- Drop Identifier Columns
identifier_cols = [
    "patient_id",
    "lesion_id"
]

df = df.drop(
    columns=identifier_cols,
    errors="ignore"
)

In [5]:
# step 4b - Drop leakage columns
leakage_cols = [
    "diagnostic",
    "biopsed"
]

df = df.drop(
    columns=leakage_cols,
    errors="ignore"
)

### STEP 5 — Define Features and Target

In [6]:

X = df.drop(
    columns="cancerous_flag"
)

y = df["cancerous_flag"]

### STEP 6 — Identify Feature Types

In [7]:

numerical_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:", len(numerical_cols))
print("Categorical Features:", len(categorical_cols))

Numerical Features: 5
Categorical Features: 19


In [8]:
print(numerical_cols)
print(categorical_cols)

['age', 'fitspatrick', 'diameter_1', 'diameter_2', 'avg_diameter']
['age_group', 'gender', 'smoke', 'drink', 'pesticide', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'background_father', 'background_mother', 'region', 'lesion_size_category', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']


### STEP 7 — Train-Test Split


In [9]:
# step 7a - Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [10]:
# step 7b -  Check the split
print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print()

print("Training target distribution")
print(y_train.value_counts(normalize=True))

print()

print("Testing target distribution")
print(y_test.value_counts(normalize=True))

Training set: (870, 24)
Testing set : (218, 24)

Training target distribution
cancerous_flag
0    0.681609
1    0.318391
Name: proportion, dtype: float64

Testing target distribution
cancerous_flag
0    0.683486
1    0.316514
Name: proportion, dtype: float64


### STEP 8 — Build the Preprocessing Pipeline

In [11]:
# step 8a - Numeric pipeline

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

numerical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

In [12]:
# step 8b -  categorical pipeline

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

In [13]:
# step 8c -  combine both pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_cols
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_cols
        )
    ],
    remainder="drop"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

### STEP 9 — Fit on Training Data Only

In [14]:
# step 9a - fit the data
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)



In [15]:
# step 9b -  verify the transformed shapes:

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print()

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (870, 24)
Processed training shape: (870, 75)

Original testing shape: (218, 24)
Processed testing shape: (218, 75)


### STEP 10 — Retrieve Feature Names

In [16]:
# step 10 - retrieved feature names

feature_names = preprocessor.get_feature_names_out()

print(f"Total processed features: {len(feature_names)}")
print(feature_names)

Total processed features: 75
['num__age' 'num__fitspatrick' 'num__diameter_1' 'num__diameter_2'
 'num__avg_diameter' 'cat__age_group_Elderly' 'cat__age_group_Middle Age'
 'cat__age_group_Older Adult' 'cat__age_group_Young Adult'
 'cat__gender_FEMALE' 'cat__gender_MALE' 'cat__smoke_f' 'cat__smoke_t'
 'cat__drink_f' 'cat__drink_t' 'cat__pesticide_f' 'cat__pesticide_t'
 'cat__skin_cancer_history_f' 'cat__skin_cancer_history_t'
 'cat__cancer_history_f' 'cat__cancer_history_t' 'cat__has_piped_water_f'
 'cat__has_piped_water_t' 'cat__has_sewage_system_f'
 'cat__has_sewage_system_t' 'cat__background_father_AUSTRIA'
 'cat__background_father_BRAZIL' 'cat__background_father_GERMANY'
 'cat__background_father_ISRAEL' 'cat__background_father_ITALY'
 'cat__background_father_NETHERLANDS' 'cat__background_father_POLAND'
 'cat__background_father_POMERANIA' 'cat__background_father_PORTUGAL'
 'cat__background_father_UNK' 'cat__background_father_UNKNOWN'
 'cat__background_mother_BRAZIL' 'cat__background_m

In [17]:
# step 10b -  Create DataFrames:

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

X_train_processed_df.head()

,num__age,num__fitspatrick,num__diameter_1,num__diameter_2,num__avg_diameter,cat__age_group_Elderly,cat__age_group_Middle Age,cat__age_group_Older Adult,cat__age_group_Young Adult,cat__gender_FEMALE,...,cat__grew_f,cat__grew_t,cat__hurt_f,cat__hurt_t,cat__changed_f,cat__changed_t,cat__bleed_f,cat__bleed_t,cat__elevation_f,cat__elevation_t
151,1.246696,-0.851742,-0.725383,-0.728161,-0.737225,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
552,-0.531178,-0.851742,-0.725383,-0.728161,-0.737225,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
785,-0.224648,0.688831,0.239576,0.196117,0.224253,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
415,0.817554,-0.851742,-0.725383,-0.728161,-0.737225,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
938,-0.224648,2.999692,-0.587532,-0.543305,-0.576979,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [18]:
# step 10c -  Save the Preprocessor and Split Data
MODELS_DIR = os.path.join("..", "Models")
PROCESSED_DIR = os.path.join("..", "data", "processed")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [19]:
# step 10d -  Save the fitted preprocessor:
joblib.dump(
    preprocessor,
    os.path.join(MODELS_DIR, "preprocessor.joblib")
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [20]:
# step 10e - save split data

X_train.to_csv(
    os.path.join(PROCESSED_DIR, "X_train.csv"),
    index=False
)

X_test.to_csv(
    os.path.join(PROCESSED_DIR, "X_test.csv"),
    index=False
)

y_train.to_csv(
    os.path.join(PROCESSED_DIR, "y_train.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(PROCESSED_DIR, "y_test.csv"),
    index=False
)

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


## Preprocessing Summary

- Identifier columns were removed.
- `diagnostic` was excluded to prevent target leakage.
- The data was split before preprocessing.
- Numerical features were median-imputed and standardised.
- Categorical features were mode-imputed and one-hot encoded.
- The fitted preprocessing object was saved for modelling and deployment.



In [21]:
print(X.columns.tolist())

['age', 'age_group', 'gender', 'smoke', 'drink', 'pesticide', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'background_father', 'background_mother', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'avg_diameter', 'lesion_size_category', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']
